In [1]:
import torch
print(torch.cuda.is_available())

True


In [3]:
import os
import shutil

source_dir = "small_dataset"
target_dir = "dataset_ready"

os.makedirs(f"{target_dir}/cancer", exist_ok=True)
os.makedirs(f"{target_dir}/non_cancer", exist_ok=True)

for patient in os.listdir(source_dir):
    patient_path = os.path.join(source_dir, patient)

    if not os.path.isdir(patient_path):
        continue

    for label in ['0', '1']:
        label_path = os.path.join(patient_path, label)

        if not os.path.exists(label_path):
            continue

        for img in os.listdir(label_path):
            src = os.path.join(label_path, img)

            if label == '0':
                dst = os.path.join(target_dir, "non_cancer", img)
            else:
                dst = os.path.join(target_dir, "cancer", img)

            shutil.copy(src, dst)

In [5]:
import os
print(len(os.listdir("E:/dataset_ready/cancer")))
print(len(os.listdir("E:/dataset_ready/non_cancer")))

0
0


In [7]:
import os
import shutil
import random

source_dir = "dataset_ready"
train_dir = "dataset_final/train"
val_dir = "dataset_final/val"

classes = ["cancer", "non_cancer"]

for cls in classes:
    os.makedirs(f"{train_dir}/{cls}", exist_ok=True)
    os.makedirs(f"{val_dir}/{cls}", exist_ok=True)

    files = os.listdir(f"{source_dir}/{cls}")
    random.shuffle(files)

    split = int(0.8 * len(files))  # 80-20 split

    train_files = files[:split]
    val_files = files[split:]

    for f in train_files:
        shutil.copy(f"{source_dir}/{cls}/{f}", f"{train_dir}/{cls}/{f}")

    for f in val_files:
        shutil.copy(f"{source_dir}/{cls}/{f}", f"{val_dir}/{cls}/{f}")

In [33]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_data = datasets.ImageFolder("dataset_final/train", transform=transform)
val_data = datasets.ImageFolder("dataset_final/val", transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)

In [16]:
print(len(train_data), len(val_data))
print(train_data.classes)

10469 2618
['cancer', 'non_cancer']


In [34]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.fc = nn.Linear(model.fc.in_features, 2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [35]:
import torch.optim as optim

weights = [10469/2172, 10469/8297]
weights = torch.tensor(weights).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [36]:
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss:.4f}")

Epoch 1, Loss: 95.2022
Epoch 2, Loss: 54.9976
Epoch 3, Loss: 28.3884
Epoch 4, Loss: 21.3061
Epoch 5, Loss: 12.3437
Epoch 6, Loss: 16.0254
Epoch 7, Loss: 11.2160
Epoch 8, Loss: 9.5536
Epoch 9, Loss: 12.8343
Epoch 10, Loss: 7.6293


In [37]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy: {100 * correct / total:.2f}%")

Accuracy: 92.40%


In [26]:
from collections import Counter

print(Counter([label for _, label in train_data]))

Counter({1: 8297, 0: 2172})


In [25]:
images, labels = next(iter(val_loader))
outputs = model(images.to(device))

_, preds = torch.max(outputs, 1)

print(preds[:10])
print(labels[:10])

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])


In [30]:
pip install scikit-learn

     ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
     - -------------------------------------- 0.3/8.9 MB 7.9 MB/s eta 0:00:02
     -- ------------------------------------- 0.6/8.9 MB 6.8 MB/s eta 0:00:02
     ---- ----------------------------------- 0.9/8.9 MB 6.6 MB/s eta 0:00:02
     ----- ---------------------------------- 1.3/8.9 MB 6.9 MB/s eta 0:00:02
     ------- -------------------------------- 1.7/8.9 MB 7.2 MB/s eta 0:00:01
     --------- ------------------------------ 2.1/8.9 MB 7.9 MB/s eta 0:00:01
     ----------- ---------------------------- 2.5/8.9 MB 7.6 MB/s eta 0:00:01
     ------------- -------------------------- 2.9/8.9 MB 8.0 MB/s eta 0:00:01
     -------------- ------------------------- 3.3/8.9 MB 7.6 MB/s eta 0:00:01
     ---------------- ----------------------- 3.7/8.9 MB 7.9 MB/s eta 0:00:01
     ------------------ --------------------- 4.1/8.9 MB 8.0 MB/s eta 0:00:01
     -------------------- ------------------- 4.5/8.9 MB 7.8 MB/s eta 0

In [38]:
from sklearn.metrics import classification_report

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds))

              precision    recall  f1-score   support

           0       0.83      0.79      0.81       543
           1       0.95      0.96      0.95      2075

    accuracy                           0.92      2618
   macro avg       0.89      0.88      0.88      2618
weighted avg       0.92      0.92      0.92      2618



In [39]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(all_labels, all_preds))

[[ 431  112]
 [  87 1988]]
